# CIFAR-10 High-Accuracy CNN — VS Code Notebook

Yeh notebook aap ki local CIFAR-10 archive file ko use karti hai. VS Code mein `.ipynb` file open karein, upper-right se apna `.venv` kernel select karein, aur **Run All** karein.

Pehle VS Code terminal mein yeh chalna chahiye: `python -m pip install -r outputs/requirements.txt`. GPU ho to PyTorch CUDA detect karke automatically use karega; warna CPU use hoga.

In [ ]:
import os
import random
import tarfile
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

# ---- Settings ----
DATA_ARCHIVE = Path('C:/Users/GeoComputer/Downloads/cifar-10-python.tar.gz')
DATA_DIR = Path.cwd() / 'cifar10_data'
CHECKPOINT_PATH = Path.cwd() / 'best_cifar10_resnet18.pt'

SEED = 42
EPOCHS = 150          # Quick test ke liye 10 ya 20; high accuracy ke liye 150
BATCH_SIZE = 128
LEARNING_RATE = 0.1
MIN_LR = 1e-5
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.1
VALIDATION_RATIO = 0.10
# Windows + VS Code par 0 zyada reliable hai. Training stable ho to 2 kar sakte hain.
NUM_WORKERS = 0 if os.name == 'nt' else 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
print('Device:', device)
if use_amp:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU mode: training kaafi slow ho sakti hai.')


In [ ]:
# Archive ko ek dafa extract karein. Agar pehle extract ho chuki hai to yeh cell usay dobara extract nahi karega.
def safe_extract(archive, destination):
    destination = destination.resolve()
    for member in archive.getmembers():
        member_path = (destination / member.name).resolve()
        try:
            member_path.relative_to(destination)
        except ValueError as error:
            raise RuntimeError(f'Unsafe archive path: {member.name}') from error
    archive.extractall(destination)

expected_folder = DATA_DIR / 'cifar-10-batches-py'
if not expected_folder.is_dir():
    if not DATA_ARCHIVE.is_file():
        raise FileNotFoundError(
            f'Archive nahi mili: {DATA_ARCHIVE}\n'
            'DATA_ARCHIVE variable mein apni file ka correct path likhein.'
        )
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {DATA_ARCHIVE} ...')
    with tarfile.open(DATA_ARCHIVE, 'r:gz') as archive:
        safe_extract(archive, DATA_DIR)

if not expected_folder.is_dir():
    raise RuntimeError('cifar-10-batches-py folder extract nahi hua. Official Python archive use karein.')

print('Dataset ready at:', DATA_DIR)


In [ ]:
# Data augmentation sirf training images par apply hoti hai.
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.15), ratio=(0.3, 3.3)),
])

evaluation_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Validation data ke liye alag object is liye banaya hai taa-ke augmentation apply na ho.
training_data = datasets.CIFAR10(DATA_DIR, train=True, transform=train_transform, download=False)
validation_data = datasets.CIFAR10(DATA_DIR, train=True, transform=evaluation_transform, download=False)
test_data = datasets.CIFAR10(DATA_DIR, train=False, transform=evaluation_transform, download=False)

validation_size = int(len(training_data) * VALIDATION_RATIO)
indices = torch.randperm(len(training_data), generator=torch.Generator().manual_seed(SEED)).tolist()
validation_indices = indices[:validation_size]
training_indices = indices[validation_size:]

loader_options = {
    'num_workers': NUM_WORKERS,
    'pin_memory': use_amp,
    'persistent_workers': NUM_WORKERS > 0,
}
train_loader = DataLoader(Subset(training_data, training_indices), batch_size=BATCH_SIZE, shuffle=True, **loader_options)
validation_loader = DataLoader(Subset(validation_data, validation_indices), batch_size=BATCH_SIZE, shuffle=False, **loader_options)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, **loader_options)

print(f'Train: {len(train_loader.dataset)} | Validation: {len(validation_loader.dataset)} | Test: {len(test_loader.dataset)}')


In [ ]:
# ResNet-18 ko 32x32 CIFAR-10 image size ke liye adapt karte hain.
model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

train_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
evaluation_criterion = nn.CrossEntropyLoss()
optimizer = SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=0.9,
    nesterov=True,
    weight_decay=WEIGHT_DECAY,
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def autocast_context():
    return torch.autocast(device_type='cuda', dtype=torch.float16) if use_amp else nullcontext()

def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=use_amp)
            labels = labels.to(device, non_blocking=use_amp)
            with autocast_context():
                logits = model(images)
                loss = evaluation_criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

print(f'Model parameters: {sum(parameter.numel() for parameter in model.parameters()):,}')


In [ ]:
# Model train karein. Highest validation accuracy wala model automatically save hoga.
best_validation_accuracy = -1.0
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=use_amp)
        labels = labels.to(device, non_blocking=use_amp)
        optimizer.zero_grad(set_to_none=True)

        with autocast_context():
            logits = model(images)
            loss = train_criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    validation_loss, validation_accuracy = evaluate(validation_loader)
    scheduler.step()
    train_accuracy = 100.0 * correct / total
    current_lr = optimizer.param_groups[0]['lr']

    print(
        f'Epoch {epoch:03d}/{EPOCHS} | train loss {running_loss / total:.4f} | '
        f'train acc {train_accuracy:.2f}% | val loss {validation_loss:.4f} | '
        f'val acc {validation_accuracy:.2f}% | lr {current_lr:.6f}'
    )

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        torch.save({
            'epoch': epoch,
            'validation_accuracy': validation_accuracy,
            'model_state_dict': model.state_dict(),
            'class_names': training_data.classes,
        }, CHECKPOINT_PATH)
        print(f'  Best model saved: {CHECKPOINT_PATH}')


In [ ]:
# Best validation model ko load kar ke final, unseen test set par accuracy check karein.
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
test_loss, test_accuracy = evaluate(test_loader)

print('Training complete')
print(f"Best validation accuracy: {checkpoint['validation_accuracy']:.2f}% (epoch {checkpoint['epoch']})")
print(f'Final test loss: {test_loss:.4f}')
print(f'Final test accuracy: {test_accuracy:.2f}%')
print(f'Saved model: {CHECKPOINT_PATH}')


In [ ]:
# Optional: har class ki test accuracy dekhain.
class_correct = [0] * 10
class_total = [0] * 10
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=use_amp)
        labels = labels.to(device, non_blocking=use_amp)
        with autocast_context():
            predictions = model(images).argmax(dim=1)
        for label, prediction in zip(labels.cpu().tolist(), predictions.cpu().tolist()):
            class_total[label] += 1
            class_correct[label] += int(label == prediction)

for index, class_name in enumerate(training_data.classes):
    print(f'{class_name:12s}: {100.0 * class_correct[index] / class_total[index]:.2f}%')
